# Логистическая регрессия — мини-проект

Сквозной мини-проект на реальных данных. Повторяет ключевые темы урока в одном пайплайне.


In [ ]:
# Setup
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
from sklearn.calibration import calibration_curve
from sklearn.datasets import fetch_openml, load_breast_cancer, make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (, average_precision_score, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)


## Постановка задачи `(intro)`


Два реальных сценария:

1. **Breast Cancer Wisconsin** — сбалансированная бинарная классификация: Pipeline, регуляризация `C`, ROC/PR, калибровка.
2. **German Credit (OpenML)** — дисбаланс классов: ловушка Accuracy, `class_weight`, PR-кривая и порог.


## Часть 1 — данные Breast Cancer `(eda)`


30 числовых признаков измерений опухоли. Класс 0 — злокачественная, 1 — доброкачественная (sklearn encoding).


In [ ]:
warnings.filterwarnings("ignore")


    RocCurveDisplay, PrecisionRecallDisplay,
    classification_report, ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid", font_scale=1.05)

bc = load_breast_cancer(as_frame=True)
X_bc = bc.data
y_bc = bc.target
feature_names_bc = bc.feature_names

print("Размер:", X_bc.shape)
print("Баланс классов:\n", y_bc.value_counts(normalize=True).round(3))
display(X_bc.describe().iloc[:4].round(2))


## Baseline: Pipeline и predict_proba `(example)`


LogReg чувствительна к масштабу из-за L2-штрафа. Главный выход — **вероятности**, не только метки.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.25, random_state=42, stratify=y_bc
)

pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=2000, random_state=42),
)
pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
pred = pipe.predict(X_test)

print(classification_report(y_test, pred, target_names=bc.target_names, digits=3))

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax, cmap="Blues")
ax.set_title("Confusion matrix (порог 0.5)")
plt.tight_layout()
plt.show()


## Регуляризация: параметр C `(experiment)`


Большое `C` — слабая регуляризация (крупные веса). Малое `C` — сильное сжатие.


In [ ]:
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
rows = []
for C in C_values:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000, random_state=42))
    m.fit(X_train, y_train)
    lr = m.named_steps["logisticregression"]
    rows.append({
        "C": C,
        "accuracy test": m.score(X_test, y_test),
        "norm_w": float(np.linalg.norm(lr.coef_)),
        "max_abs_w": float(np.max(np.abs(lr.coef_))),
    })

display(pd.DataFrame(rows).round(4))


## ROC, PR и калибровка `(viz)`


На сбалансированных данных ROC информативна. Reliability diagram проверяет, совпадают ли вероятности с частотами.


In [ ]:
pipe.fit(X_train, y_train)
proba = pipe.predict_proba(X_test)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0])
axes[0].set_title("ROC")

PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1])
axes[1].set_title("PR (сбалансировано)")

frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=8, strategy="uniform")
axes[2].plot([0, 1], [0, 1], "k--", label="идеал")
axes[2].plot(mean_pred, frac_pos, "o-", color="crimson", label="LogReg")
axes[2].set_xlabel("средняя P(y=1)")
axes[2].set_ylabel("доля класса 1")
axes[2].set_title("Калибровка")
axes[2].legend()
plt.tight_layout()
plt.show()


## Интерпретация: odds и log-odds `(example)`


Линейная комбинация $z = w·x + b$ — log-odds. Увеличение признака на 1 умножает odds на $e^{w_j}$ (при прочих равных).


In [ ]:
lr = pipe.named_steps["logisticregression"]
w = lr.coef_.ravel()
top_idx = np.argsort(np.abs(w))[-8:]

plt.figure(figsize=(8, 4))
plt.barh(np.array(feature_names_bc)[top_idx], w[top_idx], color="steelblue")
plt.xlabel("вес (log-odds, после scaler)")
plt.title("Топ-8 признаков по |w|")
plt.tight_layout()
plt.show()

j = top_idx[-1]
print(f"Признак: {feature_names_bc[j]}")
print(f"  w = {w[j]:+.3f} -> odds x {np.exp(w[j]):.2f} при +1 sigma после scaler")


## Часть 2 — German Credit: дисбаланс `(eda)`


Кредитная история: класс «плохой» клиент встречается реже. Accuracy может обмануть.


In [ ]:
credit = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
df_cr = credit.frame
y_cr = (df_cr["class"] == "bad").astype(int)
X_cr = pd.get_dummies(df_cr.drop(columns=["class"]), drop_first=True)

print("Размер:", X_cr.shape)
print("Доля 'bad':", y_cr.mean().round(3))

fig, ax = plt.subplots(figsize=(4, 3))
y_cr.value_counts().plot(kind="bar", ax=ax, color=["steelblue", "crimson"])
ax.set_xticklabels(["good (0)", "bad (1)"], rotation=0)
ax.set_title("Несбалансированные классы")
plt.tight_layout()
plt.show()


## Ловушка Accuracy и class_weight `(experiment)`


Модель «всегда good» даёт высокий accuracy. `class_weight='balanced'` усиливает штраф за ошибки на минорном классе.


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cr, y_cr, test_size=0.3, random_state=42, stratify=y_cr
)

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_tr, y_tr)
print(f"Dummy accuracy: {dummy.score(X_te, y_te):.3f}\n")

for cw, label in [(None, "без весов"), ("balanced", "balanced")]:
    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight=cw, max_iter=2000, random_state=42),
    )
    m.fit(X_tr, y_tr)
    y_pred = m.predict(X_te)
    print(f"--- {label} ---")
    print(classification_report(y_te, y_pred, target_names=["good", "bad"], digits=3))


## PR-кривая и выбор порога `(viz)`


При дисбалансе смотрим **PR-AUC**. Порог 0.5 не обязан быть оптимальным — сдвигаем по PR-кривой.


In [ ]:
m = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42),
)
m.fit(X_tr, y_tr)
proba_cr = m.predict_proba(X_te)[:, 1]

prec, rec, thresholds = precision_recall_curve(y_te, proba_cr)
ap = average_precision_score(y_te, proba_cr)


fig, axes = plt.subplots(1, 2, figsize=(11, 4))

PrecisionRecallDisplay.from_predictions(y_te, proba_cr, ax=axes[0])
axes[0].set_title(f"PR-кривая (AP={ap:.3f})")

rows_thr = []
for thr in [0.5, 0.35, 0.25]:
    y_p = (proba_cr >= thr).astype(int)
    rows_thr.append({
        "threshold": thr,
        "precision": precision_score(y_te, y_p, zero_division=0),
        "recall": recall_score(y_te, y_p, zero_division=0),
    })
thr_df = pd.DataFrame(rows_thr)
x = np.arange(len(thr_df))
w = 0.35
axes[1].bar(x - w / 2, thr_df["precision"], width=w, label="precision")
axes[1].bar(x + w / 2, thr_df["recall"], width=w, label="recall")
axes[1].set_xticks(x)
axes[1].set_xticklabels([str(t) for t in thr_df["threshold"]])
axes[1].set_xlabel("порог")
axes[1].set_title("Precision / Recall vs порог")
axes[1].legend()
plt.tight_layout()
plt.show()
display(thr_df.round(3))


## Ограничение: линейная граница `(viz)`


LogReg — один нейрон с сигмоидой. На XOR-подобных данных линейная граница не справляется.


In [ ]:
X_xor, _ = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, class_sep=1.0, random_state=42,
)
y_xor = ((X_xor[:, 0] > 0) ^ (X_xor[:, 1] > 0)).astype(int)

m_xor = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500, random_state=42))
m_xor.fit(X_xor, y_xor)

xx, yy = np.meshgrid(
    np.linspace(X_xor[:, 0].min() - 0.5, X_xor[:, 0].max() + 0.5, 200),
    np.linspace(X_xor[:, 1].min() - 0.5, X_xor[:, 1].max() + 0.5, 200),
)
Z = m_xor.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.7)
plt.colorbar(label="P(y=1)")
plt.scatter(X_xor[y_xor == 0, 0], X_xor[y_xor == 0, 1], c="blue", edgecolors="k", s=40)
plt.scatter(X_xor[y_xor == 1, 0], X_xor[y_xor == 1, 1], c="red", edgecolors="k", s=40)
plt.title(f"XOR: accuracy={m_xor.score(X_xor, y_xor):.2f}")
plt.xlabel("x₁")
plt.ylabel("x₂")
plt.tight_layout()
plt.show()


## Чек-лист мини-проекта `(summary)`


1. LogReg — базлайн для табличной классификации.
2. Всегда `Pipeline` + `StandardScaler`.
3. Смотреть `predict_proba`, не только accuracy.
4. При дисбалансе — PR-кривая, `class_weight`, подбор порога.
5. Калибровка: reliability diagram на hold-out.
6. Интерпретация через log-odds / odds.
7. Линейная граница — ограничение; для XOR нужны признаки или другие модели.
